In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [2]:
def getSparkSession():
    spark = SparkSession.builder.appName('students').getOrCreate()
    return spark
def load_data(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    return df
    
file1='D:/spark/datasets/Road_accidents.csv'
file2 = 'D:/spark/datasets/Road_accidents_time.csv'

spark = getSparkSession()
df1 = load_data(spark, file1)
df2 = load_data(spark, file2)

In [3]:
def prepare_merged_df(df1, df2):

    return df1.join(
        df2,
        on=["YEAR", "STATE/UT"],
        how="inner")
    
df = prepare_merged_df(df1, df2)

In [4]:
monthcols = ['JANUARY', 'FEBRUARY', 'MARCH', 'APRIL', 'MAY', 'JUNE', 'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'NOVEMBER', 'DECEMBER']

In [5]:
# 1. Add column 'Total', giving sum of accidents in each month .
df1 = df.agg(*[F.sum(c).alias(c) for c in monthcols])
df1.show()

+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
|JANUARY|FEBRUARY| MARCH| APRIL|   MAY|  JUNE|  JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
| 482070|  458567|485333|479021|520893|472912|439631|437600|   434553| 454266|  456464|  473199|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+



In [23]:
# 2. Which state has highest number of accidents in year 2013?

from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.functions import col

res = (
    df.withColumn('total', reduce(lambda a, b: a + b, [F.col(c) for c in monthcols]))
    .filter(col('YEAR') == 2013)
    .groupBy("STATE/UT")
    .agg(F.sum('total').alias('total_acc'))
    .orderBy(F.col('total_acc').desc())
)
res.show(1)

+----------+---------+
|  STATE/UT|total_acc|
+----------+---------+
|Tamil Nadu|    66238|
+----------+---------+
only showing top 1 row



In [27]:
# 3. Find the average monthly accidents for each state. 

res3 = (
    res.withColumn('avg_month', col('total_acc')/12)
    .groupBy("STATE/UT")
    .agg(F.avg('avg_month').alias('avgmonth'))
    .orderBy(F.col('avgmonth').desc())
)
res3.show()

+---------------+------------------+
|       STATE/UT|          avgmonth|
+---------------+------------------+
|     Tamil Nadu| 5519.833333333333|
|    Maharashtra|           3655.25|
| Andhra Pradesh|3587.3333333333335|
|      Karnataka|           3299.25|
|         Kerala|2934.5833333333335|
| Madhya Pradesh|2836.1666666666665|
|  Uttar Pradesh|2164.5833333333335|
|        Gujarat|           2086.25|
|      Rajasthan|            1965.5|
|    West Bengal|1379.0833333333333|
|   Chhattisgarh|1138.0833333333333|
|        Haryana|            872.75|
|          Bihar| 849.8333333333334|
|         Odisha| 806.6666666666666|
|     Delhi (Ut)|            630.75|
|          Assam| 600.9166666666666|
|Jammu & Kashmir| 537.9166666666666|
|         Punjab| 526.9166666666666|
|      Jharkhand| 430.0833333333333|
|            Goa|            357.75|
+---------------+------------------+
only showing top 20 rows



In [33]:
# 4. Which month has highest accidents?
import builtins

totals_row = df1.first()
month_sums = [(m, totals_row[m]) for m in monthcols]

highest_month = builtins.max(month_sums, key=lambda x: x[1])
highest_month

('MAY', 520893)

In [44]:
# 5. Which time slot (like 0-3, 3-6 etc) has more accidents? What about 'Karnataka' state?

# 1. Define the mapping (new column names must be strings)
column_mapping = {
    '0-3 hrs. (Night)': '1',
    '3-6 hrs. (Night)': '2',
    '6-9 hrs (Day)': '3',
    '9-12 hrs (Day)': '4',
    '12-15 hrs (Day)': '5',
    '15-18 hrs (Day)': '6',
    '18-21 hrs (Night)': '7',
    '21-24 hrs (Night)': '8'
}

df2 = df.withColumnsRenamed(column_mapping)
df2.show()

+----+--------------+-------+--------+-----+-----+----+----+----+------+---------+-------+--------+--------+----+----+----+----+----+----+----+----+
|YEAR|      STATE/UT|JANUARY|FEBRUARY|MARCH|APRIL| MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|   1|   2|   3|   4|   5|   6|   7|   8|
+----+--------------+-------+--------+-----+-----+----+----+----+------+---------+-------+--------+--------+----+----+----+----+----+----+----+----+
|2001| A & N Islands|      8|      23|   15|   15|  14|  19|  14|    19|        7|     12|      13|      22|   2|   6|  29|  40|  39|  40|  18|   7|
|2002| A & N Islands|     12|      10|   14|   16|  10|   7|  16|    11|       23|     21|      11|      17|   2|   6|  22|  41|  33|  33|  23|   8|
|2003| A & N Islands|     19|      13|   15|   13|  13|  12|   8|    16|       17|     25|      14|      15|   2|   8|  31|  35|  28|  36|  25|  15|
|2004| A & N Islands|     21|      14|   22|   17|  13|  18|  16|    19|       16|     20|      15|      2

In [52]:
hcols = [
    '1',
    '2',
    '3',
    '4',
    '5',
    '6',
    '7',
    '8'
    ]

In [54]:
unpivoted = df2.unpivot(
    ids=["STATE/UT", "YEAR"],
    values=hcols,
    variableColumnName="time_slot",
    valueColumnName="accident_count"
)

unpivoted.show(10, truncate=False)

+-------------+----+---------+--------------+
|STATE/UT     |YEAR|time_slot|accident_count|
+-------------+----+---------+--------------+
|A & N Islands|2001|1        |2             |
|A & N Islands|2001|2        |6             |
|A & N Islands|2001|3        |29            |
|A & N Islands|2001|4        |40            |
|A & N Islands|2001|5        |39            |
|A & N Islands|2001|6        |40            |
|A & N Islands|2001|7        |18            |
|A & N Islands|2001|8        |7             |
|A & N Islands|2002|1        |2             |
|A & N Islands|2002|2        |6             |
+-------------+----+---------+--------------+
only showing top 10 rows



In [56]:
overall_slot_totals = (
    unpivoted.groupBy("time_slot")
              .agg(F.sum("accident_count").alias("total_accidents"))
              .orderBy(F.col("total_accidents").desc())
)

overall_slot_totals.show(truncate=False)

+---------+---------------+
|time_slot|total_accidents|
+---------+---------------+
|6        |905409         |
|7        |872101         |
|4        |858512         |
|5        |823167         |
|3        |670989         |
|8        |600548         |
|2        |474344         |
|1        |389439         |
+---------+---------------+



In [57]:
karnataka_slot_totals = (
    unpivoted.filter(F.col("STATE/UT") == "Karnataka")
              .groupBy("time_slot")
              .agg(F.sum("accident_count").alias("total_accidents"))
              .orderBy(F.col("total_accidents").desc())
)

karnataka_slot_totals.show(truncate=False)

+---------+---------------+
|time_slot|total_accidents|
+---------+---------------+
|6        |93126          |
|7        |92944          |
|4        |88770          |
|5        |87973          |
|8        |65768          |
|3        |64457          |
|2        |49225          |
|1        |42498          |
+---------+---------------+



In [64]:
# 6. Which state has more accidents from year 2001 to 2014?

res6 = (
    df.withColumn('total', reduce(lambda a, b: a + b, [F.col(c) for c in monthcols]))
    .filter(F.col("YEAR")>=2001).filter(F.col("YEAR")<=2014)
    .groupBy("STATE/UT")
    .agg(F.sum('total').alias('total_acc'))
    .orderBy(F.col('total_acc').desc())
)
res6.show(1)

+----------+---------+
|  STATE/UT|total_acc|
+----------+---------+
|Tamil Nadu|   852073|
+----------+---------+
only showing top 1 row



In [86]:
rest = (
    df.withColumn('total', reduce(lambda a, b: a + b, [F.col(c) for c in monthcols]))
    .filter(F.col("YEAR")>=2001).filter(F.col("YEAR")<=2014)
    .groupBy("STATE/UT")
    .agg(F.round(avg('total'), 2).alias('avg_acc'))
    .orderBy('avg_acc', ascending=False)
).show()

+---------------+--------+
|       STATE/UT| avg_acc|
+---------------+--------+
|     Tamil Nadu|60862.36|
|    Maharashtra|45829.57|
|      Karnataka|41768.64|
| Andhra Pradesh|39058.64|
|         Kerala| 37011.5|
| Madhya Pradesh|28143.36|
|      Rajasthan|23118.07|
|        Gujarat|21451.79|
|  Uttar Pradesh|19609.14|
|    West Bengal|13526.79|
|        Haryana| 9422.93|
|     Delhi (Ut)| 8535.69|
|   Chhattisgarh|  8415.5|
|         Odisha| 8231.21|
|          Bihar| 6617.71|
|Jammu & Kashmir| 5915.71|
|          Assam| 4408.43|
|         Punjab| 3343.14|
|      Jharkhand| 3300.64|
|            Goa| 3209.36|
+---------------+--------+
only showing top 20 rows



In [91]:
# 7. 7. List states whose accidents number in year 2014 is less than state average from 2001 to 2014.

res7 = (
    df.withColumn('total', reduce(lambda a, b: a + b, [F.col(c) for c in monthcols]))
      .filter(F.col("YEAR") >= 2001)
      .filter(F.col("YEAR") <= 2014)
      .groupBy("STATE/UT")
      .agg(F.avg('total').alias('avg_acc'))
)

state_2014 = (
    df.withColumn('total', reduce(lambda a, b: a + b, [F.col(c) for c in monthcols]))
      .filter(F.col("YEAR") == 2014)
      .select(
          F.col("STATE/UT"),
          F.col("total").alias("acc_2014") 
      )
)

result = (
    state_2014.join(res7, on="STATE/UT", how="inner")
              .filter(F.col("acc_2014") < F.col("avg_acc"))
              .orderBy(F.col("avg_acc").desc())
)

result.show()

+-----------------+--------+------------------+
|         STATE/UT|acc_2014|           avg_acc|
+-----------------+--------+------------------+
|      Maharashtra|   44382| 45829.57142857143|
|           Kerala|   35872|           37011.5|
|  Jammu & Kashmir|    5778| 5915.714285714285|
|       Puducherry|     671|            1513.5|
|      Uttarakhand|     801|1299.4285714285713|
|          Tripura|     716| 752.4285714285714|
|       Chandigarh|     366|457.92857142857144|
|Arunachal Pradesh|     185|242.07142857142858|
|           Sikkim|     130|             189.0|
|         Nagaland|      28|55.785714285714285|
|      Daman & Diu|      39| 45.92857142857143|
+-----------------+--------+------------------+



In [ ]:
# 8. Which month has more accidents over the year?  